# <a id='toc1_'></a>[Bladder Function - Infants (before year 1) | Recursive Feature Elimination - logistic regression creatinine features](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [Bladder Function - Infants (before year 1) | Recursive Feature Elimination - logistic regression creatinine features](#toc1_)    
  - [Configuration & data](#toc1_1_)    
    - [Visualisation of Nadir creatinine values coloured by bladder dysfunction](#toc1_1_1_)    
    - [Visualisation of logistic regression for Bladder dysfunction based on Nadir Creatinine](#toc1_1_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[Configuration & data](#toc0_)

In [ ]:
# logging setup
from puv.logging_setup import setup
setup()  # or DEBUG if you want more detail

import logging
logger = logging.getLogger(__name__)
logger.info("Logging initialized.")

In [ ]:
# Environment setup
from pathlib import Path

BASE = Path('/workspaces/CODESPACE/').resolve()

# Define paths for data and analysis directories & ensure key folders exist
PREP = BASE / "data" / "prep"
INT = BASE / "data" / "int" 
RES = BASE / "results" / "eda" 

for p in [PREP, INT]:
    p.mkdir(parents=True, exist_ok=True)

logger.info("Project base set to: %s", BASE)

In [ ]:
# Import dependencies
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({
    "font.size": 12, "axes.titlesize": 15, "axes.labelsize": 13,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 12,
})


In [ ]:
# load data
df_all = pd.read_excel(PREP / "data_all.xlsx")

df_crea = df_all[['TUR age (first)', 'Nadir creatinine', 'Bladder dysfunction']].copy()

# Uncomment to display the columns of the renal function dataset
# display(sorted(df_crea.columns.to_list()))

In [ ]:
# filter out patients without bladder dysfunction data
print ("before filtering: ",  len (df_crea))

# Remove rows with missing values in the 'Bladder dysfunction' column
df_crea = df_crea[df_crea['Bladder dysfunction'].notnull()].reset_index(drop=True)

# filter out non-infants
print ("after filtering: ",  len (df_crea))

# create dataframe for nadir creatinine and bladder dysfunction
df_bladder_dys = df_crea[['Nadir creatinine', 'Bladder dysfunction']].copy().dropna()


### <a id='toc1_1_1_'></a>[Visualisation of Nadir creatinine values coloured by bladder dysfunction](#toc0_)

In [ ]:
fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(16, 8),
    sharex=True,
    gridspec_kw={"hspace": 0.2}  # minimal vertical gap
)

# Common bins
n_crea_range = df_bladder_dys['Nadir creatinine']
bins = np.linspace(n_crea_range.min(), n_crea_range.max(), 100)

color_map = {
    0: "#ADD8E6",  # blue
    1: "#FF0000",  # red
}

category_order = sorted(df_bladder_dys["Bladder dysfunction"].unique())

# Plot each category on its own horizontal panel
for ax, label in zip(axes, category_order):
    subset = df_bladder_dys[df_bladder_dys['Bladder dysfunction'] == label]

    ax.hist(
        subset['Nadir creatinine'],
        bins=bins,
        edgecolor='black',
        alpha=0.7,
        color=color_map[label]
    )

    ax.set_ylabel("Frequency", fontsize=15)
    label_txt = "Yes" if label == 1 else "No"
    ax.set_title(f"Bladder failure: {label_txt}", fontsize=18, loc="right")



# Shared x-axis
axes[-1].set_xlabel('Nadir creatinine [μmol/L]', fontsize=18)
axes[-1].set_xticks(np.arange(0, 310, step=10))



# Global title
fig.suptitle(
    'Distribution of Nadir creatinine levels by bladder dysfunction',
    fontsize=22,
    y=0.95
)

plt.savefig(
    RES / 'Bladder_Nadir_creatinine_split_distributions.png',
    format='png',
    bbox_inches='tight'
)
plt.show()

### <a id='toc1_1_2_'></a>[Visualisation of logistic regression for Bladder dysfunction based on Nadir Creatinine](#toc0_)

In [ ]:
# Define the features and target variable

X = df_bladder_dys[['Nadir creatinine']]
y = df_bladder_dys['Bladder dysfunction']

logreg = LogisticRegression(max_iter=2000, class_weight=None)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = cross_validate(
    logreg,
    X,
    y,
    cv=cv,
    scoring={
        "pr_auc": "average_precision",       
        "balanced_accuracy": "balanced_accuracy",  
    },
    return_train_score=False,
)

pr_scores = cv_results["test_pr_auc"]
bacc_scores = cv_results["test_balanced_accuracy"]

# print("PR-AUC per fold:", pr_scores)
print("Mean PR-AUC:", pr_scores.mean().round(3), "Std eror:", (pr_scores.std(ddof=1)/np.sqrt(pr_scores.size)).round(3))
print()

# print("Balanced accuracy per fold:", bacc_scores)
print("Mean balanced accuracy:", bacc_scores.mean().round(3), "Std error:", (bacc_scores.std(ddof=1)/np.sqrt(bacc_scores.size)).round(3))



In [ ]:
# Visualisation of decision boundary with logistic regression

# Train logistic regression using only the selected feature
logreg = LogisticRegression(max_iter=2000, class_weight=None)
logreg.fit(X[["Nadir creatinine"]], y) # Train on Nadir creatinine

# Generate a range of values for Nadir creatinine (for visualization)
x_range = np.linspace(X["Nadir creatinine"].min() - 1, X["Nadir creatinine"].max() + 1, 1000)
x_range_df = pd.DataFrame(x_range, columns=["Nadir creatinine"])  # Ensure DataFrame matches feature names

# Predict probabilities
y_prob = logreg.predict_proba(x_range_df)[:, 1]  # Get probability for class 1 --> Renal function at last follow up = 1

# Find the decision boundary (where probability = 0.5)
decision_boundary = x_range[np.abs(y_prob - 0.5).argmin()]  # Closest x value where probability is 0.5

# Define jitter strength
jitter_strength = 0.02

# Add jitter to the y-values of the scatter plot (keeping points around 0 or 1)
y_jittered = y + np.random.normal(0, jitter_strength, size=y.shape)

# Plot the decision boundary
plt.figure(figsize=(16, 8))

# Color by bladder dysfunction (0/1)
category_order = sorted(df_bladder_dys["Bladder dysfunction"].unique())

color_map = {
    0: "#ADD8E6",  # blue
    1: "#FF0000",  # red
}

for label in category_order:
    mask = df_bladder_dys["Bladder dysfunction"] == label
    plt.scatter(
        X.loc[mask, "Nadir creatinine"],
        y_jittered[mask],
        color=color_map[label],
        label=f"Bladder dysfunction: {label}",
        edgecolor="k",
        alpha=0.8,
    )



plt.plot(x_range, y_prob, color='red', label="Logistic Regression Curve")
plt.axhline(0.5, color='lightgray', linestyle='--', label="Probability = 0.5")
plt.axvline(decision_boundary, color='black', linestyle='dashed', label=f"Decision Boundary ({decision_boundary:.2f})")

plt.xlim([0, 290])  # Set x-axis limits
plt.xticks(np.arange(0, 290, 10))
# Labels & legend
plt.xlabel("Nadir creatinine [μmol/L]")  
plt.ylabel("Predicted Probability")
plt.suptitle("Prediction of bladder dysfunction based on Nadir creatinine level", fontsize=22)
plt.title("Bladder dysfuction - Logistic Regression Visualization with Decision Boundary")
plt.legend()
plt.savefig(RES / "Bladder_dysfunction_logistic_regression_nadir_creatinine_decision_boundary.png", dpi=300)
plt.show()
